# Crypto Forecasters — Per-Model Evaluation V2

Evaluates **each model in its own cell** using **80/20 walk-forward** (the test that actually covers the full 20% hold-out). Run one cell, write down the result, move to the next.

Models: **N-HiTS, LightGBM, GRU, Assembly, Chronos** (TFT excluded).

Single protocol: **80/20 walk-forward** with a **7-day** horizon.
- Split: 80% train / 20% test by row position (chronological, never shuffled).
- Slide a 7-day window across the full 20%, re-fitting each step.
- Reports **mean MAPE ± std** and **day-1 vs day-7 MAPE** (useful for the paper).

**To change coin:** edit `EVAL_SYMBOL` in cell 7 (Setup) and re-run that cell + the model cells.

**Before running:** Runtime → Change runtime type → T4 GPU.

## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('No GPU — go to Runtime > Change runtime type > GPU')

## 2. Install dependencies

In [ ]:
# Quote the version specs so the shell does NOT read >= as a redirect.
!pip install -q "neuralforecast>=1.7.0" "lightgbm>=4.0.0" yfinance joblib scikit-learn scipy chronos-forecasting

# Verify the import that N-HiTS / Assembly need. If this FAILS, do
# Runtime > Restart session, then run this cell again (do NOT re-run cell 1).
try:
    import neuralforecast
    print('neuralforecast OK, version', neuralforecast.__version__)
except Exception as e:
    print('neuralforecast NOT available ->', e)

## 3. Mount Google Drive and clone/pull the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

REPO_DIR    = '/content/drive/MyDrive/capstone_project_unfc'
REPO_URL    = 'https://github.com/RocioT08/capstone_project_unfc.git'
BACKEND_DIR = os.path.join(REPO_DIR, 'backend')

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)
print('Backend dir:', BACKEND_DIR)

## 4. Configuration (fixed params for all tickers)

In [ ]:
import random
import numpy as np
import torch

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

TICKERS_WITH_FEAR_GREED = {
    'ETH-USD', 'BNB-USD', 'SOL-USD', 'XRP-USD',
    'ADA-USD', 'AVAX-USD', 'DOGE-USD',
}

DATA_START       = '2016-01-01'
CONFIDENCE_LEVEL = 0.95
HORIZON          = 7            # 7 days = what the front-end shows

# ---- walk-forward 80/20 ----------------------------------------------------
# WALK_STEP=14 -> ~45 windows (covers the whole 20% but half the time of step=7).
# Set to 7 for ~89 windows (finer, slower). ALL models use the same value so the
# comparison is fair.
WALK_STEP   = 14
MAX_WINDOWS = None             # None = cover the WHOLE 20%. Set a number to limit.

print('Horizon   :', HORIZON, 'days')
print('Walk step :', WALK_STEP, 'days')

## 5. Data helpers (yfinance)

In [ ]:
import pandas as pd
import yfinance as yf
import math

def fetch_ohlcv_yf(symbol: str, start: str = DATA_START) -> pd.DataFrame:
    ticker = yf.Ticker(symbol)
    df = ticker.history(start=start, auto_adjust=True)
    df.index = pd.to_datetime(df.index, utc=True)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].astype(float)
    df = df.sort_index().dropna()
    print(f'{symbol}: {len(df)} rows from {df.index[0].date()} to {df.index[-1].date()}')
    return df

def _compute_error_metrics(actuals, predictions):
    mae  = float(np.mean([abs(a - p) for a, p in zip(actuals, predictions)]))
    rmse = float(math.sqrt(np.mean([(a - p)**2 for a, p in zip(actuals, predictions)])))
    mape = float(np.mean([abs(a - p) / abs(a) * 100 for a, p in zip(actuals, predictions) if a != 0]))
    return {'mae': round(mae, 4), 'rmse': round(rmse, 4), 'mape': round(mape, 4)}

## 6. Import models

In [ ]:
import importlib

# Import the backend FIRST, then reload the model modules. This avoids the
# "neuralforecast is required" cache that happens when the pip install in cell 2
# runs AFTER these modules were first imported.
import neuralforecast
import analytics.forecasting.crypto.nhits_forecaster as _nh
importlib.reload(_nh)   # re-run its import block now that neuralforecast exists

from analytics.forecasting.crypto.nhits_forecaster import NHiTSForecaster, _fetch_fear_greed
from analytics.forecasting.crypto.lightgbm_forecaster import LightGBMForecaster
from analytics.forecasting.crypto.gru import GRUForecaster
from analytics.forecasting.crypto.assembly import CryptoAssemblyForecaster
from analytics.forecasting import chronos2

print('neuralforecast version  :', neuralforecast.__version__)
print('N-HiTS backend available:', _nh._NHITS_AVAILABLE)
print('Imports OK')

## 7. Setup walk-forward  👈 CHANGE THE TICKER HERE

Edit `EVAL_SYMBOL` below and re-run **only this cell** + the model cells. It loads the data, makes the 80/20 split and defines `run_walkforward()`. Results accumulate in `WF_RESULTS`.

In [ ]:
# ===========================================================================
EVAL_SYMBOL = 'BNB-USD'   # 👈 CHANGE THE TICKER HERE (BTC-USD, ETH-USD, SOL-USD, ...)
# ===========================================================================

# -- Data + Fear & Greed -----------------------------------------------------
ohlcv = fetch_ohlcv_yf(EVAL_SYMBOL)

fear_greed = None
if EVAL_SYMBOL in TICKERS_WITH_FEAR_GREED:
    try:
        fear_greed = _fetch_fear_greed(n_days=3000)
        print(f'Fear & Greed: {len(fear_greed)} rows')
    except Exception as e:
        print(f'Fear & Greed unavailable: {e}')

# -- 80/20 split by row position (chronological) -----------------------------
split_idx = int(len(ohlcv) * 0.80)
test_20   = ohlcv.iloc[split_idx:]

# Same windows for ALL models (fair comparison)
window_steps = list(range(0, len(test_20) - (HORIZON - 1), WALK_STEP))
if MAX_WINDOWS is not None:
    window_steps = window_steps[:MAX_WINDOWS]

print(f'Ticker     : {EVAL_SYMBOL}')
print(f'Train (80%): {split_idx} rows')
print(f'Test  (20%): {len(test_20)} rows')
print(f'Windows    : {len(window_steps)}  (step={WALK_STEP} days, horizon={HORIZON})')

# Accumulated results (key = (ticker, model) so coins do not get mixed)
try:
    WF_RESULTS
except NameError:
    WF_RESULTS = {}


def _fit_forecast(name, factory, train_ohlcv, periods=HORIZON):
    """Fit one model on train_ohlcv -> return its point_forecast list."""
    if name == 'Chronos':
        return chronos2.forecast(train_ohlcv['Close'], periods, CONFIDENCE_LEVEL, '1d')['point_forecast']
    model = factory()
    try:
        model.fit(train_ohlcv, fear_greed=fear_greed)   # N-HiTS / Assembly
    except TypeError:
        model.fit(train_ohlcv)                            # GRU / LightGBM
    return model.forecast(periods=periods)['point_forecast']


def run_walkforward(name, factory):
    """Run the 80/20 walk-forward for ONE model on EVAL_SYMBOL. Saves to WF_RESULTS."""
    rows, perday, errors = [], [], []
    print(f"Walk-forward: {EVAL_SYMBOL} | {name}  ({len(window_steps)} windows)\n")
    for i, step in enumerate(window_steps, 1):
        ctx_end    = split_idx + step
        actual_end = ctx_end + HORIZON
        if actual_end > len(ohlcv):
            break
        context = ohlcv.iloc[:ctx_end]
        actuals = ohlcv['Close'].iloc[ctx_end:actual_end].values
        label   = str(ohlcv.index[ctx_end].date())
        try:
            preds = _fit_forecast(name, factory, context, HORIZON)
            m = _compute_error_metrics(actuals.tolist(), preds)
            rows.append({'window': label, **m})
            ape = [abs(a - p) / abs(a) * 100 for a, p in zip(actuals, preds) if a != 0]
            if len(ape) == HORIZON:
                perday.append(ape)
            print(f"  [{i:>2}/{len(window_steps)}] {label}  MAPE={m['mape']:.4f}%")
        except Exception as e:
            errors.append(f"{label}: {type(e).__name__}: {e}")
            print(f"  [{i:>2}/{len(window_steps)}] {label}  ERROR -> {type(e).__name__}: {e}")

    df = pd.DataFrame(rows)
    if df.empty:
        print(f"\n[!] No windows succeeded for {name}. First errors:")
        for er in errors[:5]:
            print('    -', er)
        print('\nFix the error above, then re-run this cell.')
        return df

    perday_arr = np.array(perday) if perday else np.zeros((1, HORIZON))
    summary = {
        'ticker':    EVAL_SYMBOL,
        'model':     name,
        'mape_mean': round(df['mape'].mean(), 4),
        'mape_std':  round(df['mape'].std(), 4),
        'mae_mean':  round(df['mae'].mean(), 4),
        'rmse_mean': round(df['rmse'].mean(), 4),
        'mape_day1': round(float(perday_arr[:, 0].mean()), 4),
        'mape_day7': round(float(perday_arr[:, -1].mean()), 4),
        'n_windows': len(df),
    }
    WF_RESULTS[(EVAL_SYMBOL, name)] = summary
    print(f"\n{'='*56}")
    print(f"  {EVAL_SYMBOL} | {name}")
    print(f"  Mean MAPE : {summary['mape_mean']:.4f}%  +/-{summary['mape_std']:.4f}")
    print(f"  Mean MAE  : {summary['mae_mean']:.4f}")
    print(f"  Mean RMSE : {summary['rmse_mean']:.4f}")
    print(f"  MAPE day 1: {summary['mape_day1']:.4f}%   (easiest)")
    print(f"  MAPE day 7: {summary['mape_day7']:.4f}%   (hardest)")
    print(f"  Windows   : {summary['n_windows']}")
    print(f"{'='*56}")
    return df

print('\nSetup ready for', EVAL_SYMBOL, '- now run each model cell below.')

## 8. N-HiTS  <- run and write down the result

In [ ]:
df_nhits = run_walkforward(
    'N-HiTS',
    lambda: NHiTSForecaster(max_horizon=HORIZON, confidence_level=CONFIDENCE_LEVEL,
                            max_steps=500, input_size=120),
)

## 9. LightGBM  <- run and write down the result

In [ ]:
df_lgb = run_walkforward(
    'LightGBM',
    lambda: LightGBMForecaster(max_horizon=HORIZON, confidence_level=CONFIDENCE_LEVEL,
                               lags=28, n_estimators=300, learning_rate=0.03, num_leaves=63),
)

## 10. GRU  <- run and write down the result

In [ ]:
df_gru = run_walkforward(
    'GRU',
    lambda: GRUForecaster(max_horizon=HORIZON, confidence_level=CONFIDENCE_LEVEL,
                          epochs=20, mc_samples=40, lookback=60, hidden_size=128, num_layers=3),
)

## 11. Chronos (zero-shot benchmark)  <- run and write down the result

In [ ]:
df_chronos = run_walkforward('Chronos', None)   # Chronos needs no factory

## 12. Assembly (ensemble)  WARNING: SLOW — run it last / overnight

It re-trains GRU + N-HiTS + LightGBM on every window, so it takes much longer.
Uses the same windows as the others so the comparison stays fair.

In [ ]:
df_assembly = run_walkforward(
    'Assembly',
    lambda: CryptoAssemblyForecaster(
        max_horizon=HORIZON, n_splits=4, ridge_alpha=0.5, min_train_size=120,
        confidence_level=CONFIDENCE_LEVEL, use_gru=True, use_tft=False,
        gru_kwargs={'epochs': 20, 'mc_samples': 40, 'lookback': 60, 'hidden_size': 128, 'num_layers': 3},
        nhits_kwargs={'max_steps': 500, 'input_size': 120},
        lgb_kwargs={'lags': 28, 'n_estimators': 300, 'learning_rate': 0.03, 'num_leaves': 63},
    ),
)

## 13. Final comparison — models for the current ticker

In [ ]:
rows = [v for (tk, _), v in WF_RESULTS.items() if tk == EVAL_SYMBOL]
if not rows:
    print('Run at least one model cell first.')
else:
    final = pd.DataFrame(rows).set_index('model')
    final = final[['mape_mean', 'mape_std', 'mae_mean', 'rmse_mean',
                   'mape_day1', 'mape_day7', 'n_windows']].sort_values('mape_mean')
    print('=' * 70)
    print(f'  FINAL RANKING — {EVAL_SYMBOL}  (lower MAPE = better)')
    print('=' * 70)
    print(final.to_string())
    print('=' * 70)
    print('Best model:', final.index[0])

In [ ]:
# Chart: MAPE per model (+/- std) and day 1 vs day 7 — current ticker
import matplotlib.pyplot as plt

rows = [v for (tk, _), v in WF_RESULTS.items() if tk == EVAL_SYMBOL]
if rows:
    final = pd.DataFrame(rows).set_index('model').sort_values('mape_mean')
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].bar(final.index, final['mape_mean'], yerr=final['mape_std'], capsize=5, color='steelblue')
    axes[0].set_ylabel('Mean MAPE (%)'); axes[0].set_title(f'{EVAL_SYMBOL} — accuracy per model (+/- std)')

    x = np.arange(len(final)); w = 0.35
    axes[1].bar(x - w/2, final['mape_day1'], w, label='Day 1')
    axes[1].bar(x + w/2, final['mape_day7'], w, label='Day 7')
    axes[1].set_xticks(x); axes[1].set_xticklabels(final.index)
    axes[1].set_ylabel('MAPE (%)'); axes[1].set_title('Day 1 vs Day 7 (how the error grows)'); axes[1].legend()

    plt.tight_layout(); plt.show()

In [ ]:
# (Optional) Save ALL results to Drive. Skip this cell if you only take notes.
import os
OUT_DIR = '/content/drive/MyDrive/capstone_checkpoints'
os.makedirs(OUT_DIR, exist_ok=True)
if WF_RESULTS:
    pd.DataFrame(list(WF_RESULTS.values())).to_csv(
        os.path.join(OUT_DIR, 'walkforward_ranking_ALL.csv'), index=False)
    print('Saved:', os.path.join(OUT_DIR, 'walkforward_ranking_ALL.csv'))
    print('Tickers saved:', sorted({tk for tk, _ in WF_RESULTS}))